# Brain Surgery — Bước 1: Template Parity Check & Gemini Synthetic Distillation

Notebook chạy trên **Google Colab** (chỉ cần CPU hoặc T4 nhẹ), đọc `GEMINI_API_KEY` và `HF_TOKEN` từ **Colab Secrets** để:
1. Kiểm tra khớp từng ký tự giữa `render_gemma4_training_sample()` và `AutoTokenizer.apply_chat_template()` của **`haku-nekooo/gemma4-e2b-nuclear-pruned`**.
2. Chạy **Sync Smoke Test** gọi API Gemini (`gemini-2.5-pro` + `gemini-2.5-flash`) sinh thử `12` mẫu đủ 5 nhóm SFT (`A_nc_agent`..`E_dev_chat`) + `ORPO_pairs`, lọc qua `validator.py` và in preview để duyệt chất lượng.
3. *(Tùy chọn, có khóa an toàn `RUN_FULL_BATCH = False`)* Nộp **Gemini Batch API** toàn bộ ~4,000 SFT + ~600 ORPO và đẩy lên Hugging Face Private Repo.

## 1. Cài đặt môi trường & Nạp Secret (`HF_TOKEN`, `GEMINI_API_KEY`)

In [ ]:
!pip install -q --upgrade "google-genai>=1.0.0" "transformers>=5.5.0" "huggingface_hub>=0.28.0"

In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys

GITHUB_REPO_URL = "https://github.com/just-haku/brain-surgery.git"
REPO_DIR = Path("/content/brain_surgery")

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
elif not (REPO_DIR / "src" / "distill").exists():
    subprocess.run(["git", "clone", "--depth", "1", GITHUB_REPO_URL, str(REPO_DIR)], check=True)

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

HF_TOKEN = os.environ.get("HF_TOKEN")
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
try:
    from google.colab import userdata
    HF_TOKEN = HF_TOKEN or userdata.get("HF_TOKEN")
    GEMINI_API_KEY = GEMINI_API_KEY or userdata.get("GEMINI_API_KEY")
except Exception:
    pass

assert HF_TOKEN, "Thiếu HF_TOKEN (bật trong Colab Secrets)."
assert GEMINI_API_KEY, "Thiếu GEMINI_API_KEY (bật trong Colab Secrets)."
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

PRUNED_HF_REPO = "haku-nekooo/gemma4-e2b-nuclear-pruned"
WORK_DIR = Path("/content/work_step1")
WORK_DIR.mkdir(parents=True, exist_ok=True)
print("Đã nạp src/distill và các secret thành công.")

## 2. Phần (a) — Kiểm chứng Parity với `chat_template.jinja` trên Hugging Face

In [ ]:
import difflib
from transformers import AutoTokenizer
from distill.formatter import render_gemma4_training_sample, validate_gemma4_wire_format
from distill.nc_schema import build_nc_system_prompt, build_nc_user_prompt, get_nc_openai_tools

tok = AutoTokenizer.from_pretrained(PRUNED_HF_REPO, use_fast=True, token=HF_TOKEN)
tools = get_nc_openai_tools(["read_symbol", "search_text", "edit_file"])

parity_messages = [
    {"role": "system", "content": build_nc_system_prompt("coder")},
    {
        "role": "user",
        "content": build_nc_user_prompt(
            repo_map="src/math.rs: add",
            task="Fix overflow in add() keeping i32 return type.",
        ),
    },
    {
        "role": "assistant",
        "reasoning_content": "Read add() body before editing.",
        "tool_calls": [
            {
                "id": "call_1",
                "type": "function",
                "function": {
                    "name": "read_symbol",
                    "arguments": {"path": "src/math.rs", "symbol": "add"},
                },
            }
        ],
        "content": "",
    },
    {
        "role": "tool",
        "tool_call_id": "call_1",
        "name": "read_symbol",
        "content": "L10-L12:\npub fn add(a: i32, b: i32) -> i32 { a + b }",
    },
    {
        "role": "assistant",
        "reasoning_content": "Use saturating_add keeping i32 types intact.",
        "content": "Replace `a + b` with `a.saturating_add(b)` in `src/math.rs` keeping `i32`.",
    },
]

ours = render_gemma4_training_sample(parity_messages, tools=tools, enable_thinking=True, include_bos=True)
hf_text = tok.apply_chat_template(
    parity_messages,
    tools=tools,
    enable_thinking=True,
    preserve_thinking=True,
    tokenize=False,
    add_generation_prompt=False,
)

if ours["text"] != hf_text:
    diff = "".join(difflib.unified_diff(
        hf_text.splitlines(keepends=True),
        ours["text"].splitlines(keepends=True),
        fromfile="hf_apply_chat_template",
        tofile="render_gemma4_training_sample",
    ))
    raise AssertionError(f"Template parity mismatch:\n{diff}")

assert validate_gemma4_wire_format(ours["text"]) == []
enc_ids = tok.encode(ours["text"], add_special_tokens=False)
assert tok.unk_token_id not in enc_ids, "Unexpected <unk> in parity sample"

print(f"PARITY CHECK PASSED! Total tokens={len(enc_ids)}, assistant_spans={len(ours['assistant_spans'])}")
for i, (s, e) in enumerate(ours["assistant_spans"], start=1):
    print(f"--- Assistant loss span #{i} ({e - s} chars) ---")
    print(ours["text"][s:e])

## 3. Phần (b) — Sync Smoke Test (12 mẫu thực tế qua Gemini Pro & Flash)

In [ ]:
from distill.gemini_batch import parse_and_validate_results, run_sync_smoke
from distill.prompts import generate_seed_tasks

pool = generate_seed_tasks(total_sft=40, total_orpo=8, seed=42)
picked = []
for bucket_name in ("A_nc_agent", "B_bilingual_debug", "C_direct_code", "D_clarify", "E_dev_chat", "ORPO_pairs"):
    picked.extend([t for t in pool if t["bucket"] == bucket_name][:2])

print(f"Selected {len(picked)} smoke tasks:")
for t in picked:
    print(f"  {t['key']} -> {t['model']}")

smoke_raw = WORK_DIR / "smoke_raw.jsonl"
smoke_out_dir = WORK_DIR / "smoke_validated"
run_sync_smoke(picked, smoke_raw, api_key=GEMINI_API_KEY)

smoke_report = parse_and_validate_results(
    raw_jsonl_paths=[smoke_raw],
    out_sft_path=smoke_out_dir / "sft_trajectories.jsonl",
    out_orpo_path=smoke_out_dir / "orpo_pairs.jsonl",
    out_rejected_path=smoke_out_dir / "rejected.jsonl",
    tokenizer=tok,
    max_tokens=4096,
)
print("=== SMOKE VALIDATION REPORT ===")
print(json.dumps(smoke_report, indent=2, ensure_ascii=False))

for fname in ("sft_trajectories.jsonl", "orpo_pairs.jsonl", "rejected.jsonl"):
    fpath = smoke_out_dir / fname
    lines = fpath.read_text(encoding="utf-8").splitlines() if fpath.exists() else []
    print(f"\n=== PREVIEW {fname} ({len(lines)} rows) ===")
    for ln in lines[:2]:
        print(ln[:700] + ("..." if len(ln) > 700 else ""))

## 4. Phần (c) — Chuẩn bị & Nộp Gemini Batch API (Đặt `RUN_FULL_BATCH = True` sau khi duyệt Smoke)

In [ ]:
from huggingface_hub import HfApi
from distill.gemini_batch import poll_and_download_batch, submit_batch_job, write_batch_jsonl_by_model

RUN_FULL_BATCH = False  # Đổi thành True sau khi duyệt xong 12 mẫu smoke ở trên
TOTAL_SFT = 4000
TOTAL_ORPO = 600

full_tasks = generate_seed_tasks(total_sft=TOTAL_SFT, total_orpo=TOTAL_ORPO, seed=42)
batch_files = write_batch_jsonl_by_model(full_tasks, WORK_DIR / "batch_inputs")
for model_id, fpath in batch_files.items():
    n_lines = sum(1 for _ in fpath.open(encoding="utf-8"))
    print(f"Prepared {model_id}: {n_lines} requests -> {fpath}")

if RUN_FULL_BATCH:
    raw_outputs = []
    jobs = {
        model_id: submit_batch_job(fpath, model_id, f"brain-surgery-{model_id}", api_key=GEMINI_API_KEY)
        for model_id, fpath in batch_files.items()
    }
    for model_id, job_name in jobs.items():
        out_p = WORK_DIR / "batch_outputs" / f"raw_{model_id}.jsonl"
        poll_and_download_batch(job_name, out_p, poll_seconds=30, api_key=GEMINI_API_KEY)
        raw_outputs.append(out_p)

    ds_dir = WORK_DIR / "datasets"
    full_report = parse_and_validate_results(
        raw_jsonl_paths=raw_outputs,
        out_sft_path=ds_dir / "sft_trajectories.jsonl",
        out_orpo_path=ds_dir / "orpo_pairs.jsonl",
        out_rejected_path=ds_dir / "rejected.jsonl",
        tokenizer=tok,
        max_tokens=4096,
    )
    (ds_dir / "distill_report.json").write_text(
        json.dumps(full_report, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    print(json.dumps(full_report, indent=2, ensure_ascii=False))

    api = HfApi(token=HF_TOKEN)
    api.upload_folder(
        folder_path=str(ds_dir),
        path_in_repo="datasets",
        repo_id=PRUNED_HF_REPO,
        repo_type="model",
        commit_message="Step 1: upload curated SFT trajectories and ORPO preference pairs",
    )
    print(f"Uploaded datasets/ to https://huggingface.co/{PRUNED_HF_REPO}")
else:
    print("RUN_FULL_BATCH=False -> Dừng ở bước chuẩn bị file Batch JSONL.")